---
---
---

# Models

---
---
---

<br>

> | **_Language:_** python@3.12.10 |
> | - |

<br>

> | **_Source:_** `notebook/models.ipynb` |
> | - |

<br>

> | **_Configurations:_** `lib/config` |
> | - |

<br>

> | **_Libraries:_** `lib/utils` |
> | - |

<br>

---
---
---

## Dependencies

### Packages

In [ ]:
from google.cloud import bigquery
from loguru import logger
from matplotlib import pyplot as plt
from pathlib import Path
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import joblib
import json
import numpy as np
import pandas as pd
import sys
import warnings

ROOT_PATH = Path.cwd().resolve()
if ROOT_PATH.name in ["notebook"]:
    ROOT_PATH = ROOT_PATH.parent
if str(ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(ROOT_PATH))

from config import config as the_config
from utils import etl as the_etl
from utils import pipeline as the_pipeline

from lib import ml_metrics as metrics_lib
from lib.automl import AutoMLClassifier, AutoMLRegressor

# -------------------------

# Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.3f}".format)
warnings.filterwarnings("ignore")

### Constants

In [ ]:
# GCP
BQ_CLIENT = bigquery.Client(project=the_config.GCP_PROJECT)

### Paths

In [ ]:
# Paths
for path in the_config.PATHS:
    the_etl.ensure_path(path)

## DataFrame

In [ ]:
query = f"""
	SELECT * FROM `{the_config.BQ_TABLE}`
	ORDER BY TIMESTAMP(
		DATETIME(year, month, day, hour, minute, 0)
	)
"""
train_df = BQ_CLIENT.query(query).to_dataframe()
display(train_df.info())

## Training

### Feature Engineering

In [ ]:
clf_df = train_df.iloc[-the_config.ML_HYPER_SPACE["sampling"]:].copy()
clf_df, targets_clf = the_pipeline.feature_engineering_clf(
	clf_df, inference=False
)
clf_df = clf_df.assign(
	timestamp=lambda x: pd.to_datetime(
		x[["year", "month", "day", "hour", "minute"]]
	)
)
logger.debug(f"[LIST] Targets ({len(targets_clf)}): {targets_clf}")

# Feature Selection -- Classification
to_drop = [
	*the_config.CLASSIFICATION["to_drop"],
	the_config.CLASSIFICATION["targets"][0],
    "timestamp"
]
to_drop += [c for c in clf_df.columns if c.lower().startswith("conf_")]
X_clf = clf_df.drop(columns=to_drop)
y_clf = clf_df[the_config.CLASSIFICATION["targets"][0]]

In [ ]:
reg_df = train_df.iloc[-the_config.ML_HYPER_SPACE["sampling"]:].copy()
reg_df, targets_reg = the_pipeline.feature_engineering_reg(
	reg_df, inference=False
)
reg_df = reg_df.assign(
	timestamp=lambda x: pd.to_datetime(
		x[["year", "month", "day", "hour", "minute"]]
	)
)
logger.debug(f"[LIST] Targets ({len(targets_reg)}): {targets_reg}")

# Feature Selection -- Regression
to_drop = [
	*the_config.REGRESSION["to_drop"],
    *targets_reg,
	"timestamp"
]
to_drop += [c for c in reg_df.columns if c.lower().startswith("conf_")]
X_reg = reg_df.drop(columns=to_drop)
y_reg = reg_df[targets_reg]

### Train-Test Split

In [ ]:
# Train-Test Split -- Classification
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
	X_clf, y_clf,
    test_size=the_config.CLASSIFICATION["test_size"],
    shuffle=the_config.CLASSIFICATION["shuffle"],
    random_state=the_config.CLASSIFICATION["seed"]
)

logger.debug(f"{'[LIST]':<8}{'[CLF]':<6}{'Features:':<10}{X_train_clf.columns.tolist()}")
logger.debug(f"{'[LIST]':<8}{'[CLF]':<6}{'Labels:':<10}{y_train_clf.name}")
logger.debug(f"{'[TRAIN]':<8}{'[CLF]':<6}{'Shape:':<10}{y_train_clf.shape}")
logger.debug(f"{'[TEST]':<8}{'[CLF]':<6}{'Shape:':<10}{y_test_clf.shape}")

In [ ]:
# Train-Test Split -- Regression
X_train_reg = X_reg.iloc[:-the_config.INF_ROWS]
y_train_reg = y_reg.iloc[:-the_config.INF_ROWS]
X_test_reg = X_reg.iloc[-the_config.INF_ROWS:]
y_test_reg = y_reg.iloc[-the_config.INF_ROWS:]

logger.debug(f"{'[LIST]':<8}{'[REG]':<6}{'Features:':<10}{X_train_reg.columns.tolist()}")
logger.debug(f"{'[LIST]':<8}{'[REG]':<6}{'Labels:':<10}{y_train_reg.columns.tolist()}")
logger.debug(f"{'[TRAIN]':<8}{'[REG]':<6}{'Shape:':<10}{y_train_reg.shape}")
logger.debug(f"{'[TEST]':<8}{'[REG]':<6}{'Shape:':<10}{y_test_reg.shape}")

### Models

In [ ]:
# Dummy Classifier
dummy_clf = DummyClassifier(strategy="prior")
dummy_clf.fit(X_train_clf, y_train_clf)
display(dummy_clf)

# Dummy Regressor
dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train_reg, y_train_reg)
display(dummy_reg)

In [ ]:
# AutoML Classifier -- Random Forest
model_path = the_config.MODELS_PATH / "random_forest_classifier.joblib"
if model_path.exists():
    rf_clf = joblib.load(model_path)

else:
	rf_clf = AutoMLClassifier(
		**the_config.ML_HYPER_SPACE["automl_clf"],
		estimator_list=["rf"],
		seed=the_config.CLASSIFICATION["seed"]
	)
	rf_clf.fit(X_train_clf, y_train_clf)
display(rf_clf)

# AutoML Regressor -- Random Forest
model_path = the_config.MODELS_PATH / "random_forest_regressor.joblib"
if model_path.exists():
    rf_reg = joblib.load(model_path)

else:
	rf_reg = AutoMLRegressor(
		**the_config.ML_HYPER_SPACE["automl_reg"],
		estimator_list=["rf"],
		seed=the_config.REGRESSION["seed"]
	)
	rf_reg.fit(X_train_reg, y_train_reg)
display(rf_reg)

In [ ]:
# AutoML Classifier -- LGBM
model_path = the_config.MODELS_PATH / "lgbm_classifier.joblib"
if model_path.exists():
    lgbm_clf = joblib.load(model_path)

else:
	lgbm_clf = AutoMLClassifier(
		**the_config.ML_HYPER_SPACE["automl_clf"],
		estimator_list=["lgbm"],
		seed=the_config.CLASSIFICATION["seed"]
	)
	lgbm_clf.fit(X_train_clf, y_train_clf)
display(lgbm_clf)

# AutoML Regressor -- LGBM
model_path = the_config.MODELS_PATH / "lgbm_regressor.joblib"
if model_path.exists():
    lgbm_reg = joblib.load(model_path)

else:
	lgbm_reg = AutoMLRegressor(
		**the_config.ML_HYPER_SPACE["automl_reg"],
		estimator_list=["lgbm"],
		seed=the_config.REGRESSION["seed"]
	)
	lgbm_reg.fit(X_train_reg, y_train_reg)
display(lgbm_reg)

In [ ]:
# AutoML Classifier -- XGBoost
model_path = the_config.MODELS_PATH / "xgboost_classifier.joblib"
if model_path.exists():
    xgb_clf = joblib.load(model_path)

else:
	xgb_clf = AutoMLClassifier(
		**the_config.ML_HYPER_SPACE["automl_clf"],
		estimator_list=["xgboost"],
		seed=the_config.CLASSIFICATION["seed"]
	)
	xgb_clf.fit(X_train_clf, y_train_clf)
display(xgb_clf)

# AutoML Regressor -- XGBoost
model_path = the_config.MODELS_PATH / "xgboost_regressor.joblib"
if model_path.exists():
    xgb_reg = joblib.load(model_path)

else:
	xgb_reg = AutoMLRegressor(
		**the_config.ML_HYPER_SPACE["automl_reg"],
		estimator_list=["xgboost"],
		seed=the_config.REGRESSION["seed"]
	)
	xgb_reg.fit(X_train_reg, y_train_reg)
display(xgb_reg)

In [ ]:
# AutoML Classifier
model_path = the_config.MODELS_PATH / "automl_classifier.joblib"
if model_path.exists():
    automl_clf = joblib.load(model_path)

else:
	automl_clf = AutoMLClassifier(
		**the_config.ML_HYPER_SPACE["automl_clf"],
		estimator_list=[
            "rf",
            "lgbm",
            "xgboost"
        ],
		seed=the_config.CLASSIFICATION["seed"]
	)
	automl_clf.fit(X_train_clf, y_train_clf)
display(automl_clf)

# AutoML Regressor
model_path = the_config.MODELS_PATH / "automl_regressor.joblib"
if model_path.exists():
    automl_reg = joblib.load(model_path)

else:
	automl_reg = AutoMLRegressor(
		**the_config.ML_HYPER_SPACE["automl_reg"],
		estimator_list=[
			"rf",
			"lgbm",
			"xgboost"
		],
		seed=the_config.REGRESSION["seed"]
	)
	automl_reg.fit(X_train_reg, y_train_reg)
display(automl_reg)

## Evaluation

In [ ]:
models_clf = {
    "Dummy Classifier": { "model": dummy_clf, "metrics": {} },
    "Random Forest Classifier": { "model": rf_clf, "metrics": {} },
    "LGBM Classifier": { "model": lgbm_clf, "metrics": {} },
    "XGBoost Classifier": { "model": xgb_clf, "metrics": {} },
	"AutoML Classifier": { "model": automl_clf, "metrics": {} }
}
for model_name, model_info in models_clf.items():

	logger.debug(f"Model: {model_name}")
	model = model_info["model"]
	model_metrics = model_info["metrics"]

	y_true_train_clf = y_train_clf
	y_true_test_clf = y_test_clf
	y_pred_train_clf = model.predict(X_train_clf)
	y_pred_test_clf = model.predict(X_test_clf)

	model_metrics[the_config.CLASSIFICATION["targets"][0]] = {}
	metrics_lib.compute_metrics_clf(
		model_metrics[the_config.CLASSIFICATION["targets"][0]],
		y_true_train_clf,
		y_pred_train_clf,
		model_classes=[0, 1],
		set="TRAIN"
	)
	metrics_lib.compute_metrics_clf(
		model_metrics[the_config.CLASSIFICATION["targets"][0]],
		y_true_test_clf,
		y_pred_test_clf,
		model_classes=[0, 1],
		set="TEST"
	)

In [ ]:
models_reg = {
	"Dummy Regressor": { "model": dummy_reg, "metrics": {} },
    "Random Forest Regressor": { "model": rf_reg, "metrics": {} },
    "LGBM Regressor": { "model": lgbm_reg, "metrics": {} },
    "XGBoost Regressor": { "model": xgb_reg, "metrics": {} },
	"AutoML Regressor": { "model": automl_reg, "metrics": {} }
}
for model_name, model_info in models_reg.items():

	logger.debug(f"Model: {model_name}")
	model = model_info["model"]
	model_metrics = model_info["metrics"]

	y_true_train_reg = y_train_reg
	y_true_test_reg = y_test_reg
	y_pred_train_reg = pd.DataFrame(
		model.predict(X_train_reg),
		columns=y_train_reg.columns,
		index=y_train_reg.index
	)
	y_pred_test_reg = pd.DataFrame(
		model.predict(X_test_reg),
		columns=y_test_reg.columns,
		index=y_test_reg.index
	)

	for forecast in the_config.FENG_FORECASTS:

		model_metrics[forecast] = {}
		for feature in the_config.REGRESSION["targets"]:

			col = f"{feature}_lead_{forecast}"
			model_metrics[forecast][feature] = {}
			metrics_lib.compute_metrics_reg(
				model_metrics[forecast][feature],
				y_true_train_reg[[col]],
				y_pred_train_reg[[col]],
				set="TRAIN"
			)
			metrics_lib.compute_metrics_reg(
				model_metrics[forecast][feature],
				y_true_test_reg[[col]],
				y_pred_test_reg[[col]],
				set="TEST"
			)

## Results

In [ ]:
# Metrics -- Classification
rows_clf = []
for model_name, model_info in models_clf.items():

	for target, sets in model_info["metrics"].items():
		for split in ["TRAIN", "TEST"]:

			size, accuracy, precision, recall, f1_score, cm = sets[split]
			rows_clf.append({
				"Model": model_name,
				"Target": target,
				"Set": split,
				"Size": size,
				"Accuracy": accuracy,
				"Precision": precision,
				"Recall": recall,
				"F1 Score": f1_score,
				"Confusion Matrix": cm
			})
metrics_clf_df = (
    pd.DataFrame(rows_clf)
    .reset_index(drop=True)
)

display(y_train_clf.describe())
display(metrics_clf_df)

# Confusion Matrix -- Classification
for model_name in metrics_clf_df["Model"].unique():

    model_results = metrics_clf_df[
        metrics_clf_df["Model"] == model_name
    ]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    for ax, (_, row) in zip(
        axes,
        (
			model_results
			.set_index("Set")
			.loc[["TRAIN", "TEST"]]
			.reset_index()
			.iterrows()
		)
    ):

        cm = np.array(row["Confusion Matrix"])
        cm = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["dry", "rain"]
        )
        cm.plot(
            ax=ax,
            cmap="Blues",
            values_format="d"
        )
        ax.set_title(row["Set"])

    fig.suptitle(f"Confusion Matrix: {model_name}")
    fig.tight_layout()
    fig.savefig(
        the_config.MODELS_PATH /
        f"{model_name.lower().replace(' ', '_')}_cm.png",
        bbox_inches="tight",
        dpi=300
    )
    plt.show()
    plt.close(fig)

In [ ]:
# Metrics -- Regression
rows_reg = []
for model_name, model_info in models_reg.items():
    for lead, targets in model_info["metrics"].items():
        for target, sets in targets.items():
            for split in ["TRAIN", "TEST"]:

                size, mae, rmse, r2 = sets[split]
                rows_reg.append({
                    "Model": model_name,
                    "Target": target,
                    "Lead": lead,
                    "Set": split,
                    "Size": size,
                    "MAE": mae,
                    "RMSE": rmse,
                    "R2": r2,
                })                
metrics_reg_df = (
    pd.DataFrame(rows_reg)
    .sort_values(["Target", "Lead"])
    .reset_index(drop=True)
)

display(y_train_reg.describe())
display(metrics_reg_df)

## Logging

In [ ]:
# [!]
# Metrics -- Models
for models_task in [models_clf, models_reg]:
	for model_name, model_info in models_task.items():

		if model_name.lower().startswith("dummy"):
			continue

		model = model_info["model"]
		filename = model_name.lower().replace(" ", "_")
		joblib.dump(
			model,
			the_config.MODELS_PATH / f"{filename}.joblib"
		)

# Metrics -- Classification
with open(the_config.MODELS_PATH / "stats_clf.json", "w") as f:
	json.dump(
		(
			y_train_clf.to_frame().describe().T
			.to_dict(orient="index")
		),
		f, indent=4
	)
with open(the_config.MODELS_PATH / "metrics_clf.json", "w") as f:
	json.dump(
		metrics_clf_df.to_dict(orient="records"),
		f, indent=4,
		default=lambda x: x.tolist()
	)

# Metrics -- Regression
with open(the_config.MODELS_PATH / "stats_reg.json", "w") as f:
	json.dump(
		y_train_reg.describe().T
		.to_dict(orient="index"),
		f, indent=4
	)
with open(the_config.MODELS_PATH / "metrics_reg.json", "w") as f:
	json.dump(
		metrics_reg_df.to_dict(orient="records"),
		f, indent=4
	)

logger.debug(f"[OUT] {the_config.MODELS_PATH}")